# FedGNN Baseline — H&M Sampled Dataset

Federated Graph Neural Network: a two-layer bipartite GNN where user and item
embeddings are updated via neighbourhood aggregation. Item embeddings (shared
global parameter) are aggregated across clients each round; user embeddings
stay local.


In [2]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import math
import os
import itertools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


## 1. Load Data


In [4]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
TARGET_USERS     = 1000
MIN_INTERACTIONS = 20
N_QUANTILES      = 4
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cache not found in '{SAMPLED_DATA_DIR}/'. Run sampling notebook first."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded : {cache_tag}")
print(f"Users  : {n_users} | Items : {n_items}")
print(f"Train  : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")


AssertionError: Cache not found in '/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled/'. Run sampling notebook first.

## 2. Build Interaction Tensors

GNN operates on sparse (user, item) edge lists rather than dense matrices.


In [ ]:
min_rating = float(train_df["bought"].min())
max_rating = float(train_df["bought"].max())

def df_to_edges(df, min_r, max_r):
    users = torch.tensor(df["customer_id"].values, dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    return users, items, vals

train_u, train_i, train_y = df_to_edges(train_df, min_rating, max_rating)
val_u,   val_i,   val_y   = df_to_edges(val_df,   min_rating, max_rating)
test_u,  test_i,  test_y  = df_to_edges(test_df,  min_rating, max_rating)

# Dense matrices for RMSE evaluation
def make_matrix(df, n_u, n_i, min_r, max_r):
    mat   = torch.full((n_u, n_i), -1.0)
    users = torch.tensor(df["customer_id"].values, dtype=torch.long)
    items = torch.tensor(df["product_code"].values, dtype=torch.long)
    vals  = torch.tensor((df["bought"].values - min_r) / (max_r - min_r),
                         dtype=torch.float32)
    mat[users, items] = vals
    return mat

train_matrix = make_matrix(train_df, n_users, n_items, min_rating, max_rating)
val_matrix   = make_matrix(val_df,   n_users, n_items, min_rating, max_rating)
test_matrix  = make_matrix(test_df,  n_users, n_items, min_rating, max_rating)
print(f"Train edges: {len(train_u)} | Val: {len(val_u)} | Test: {len(test_u)}")


## 3. FedGNN Model

Each forward pass aggregates neighbour embeddings over the observed edges,
then predicts ratings via inner product. Item embeddings are the shared
global parameter; user embeddings are local.


In [ ]:
class FedGNN(nn.Module):
    def __init__(self, n_users, n_items, K):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, K)
        self.item_emb = nn.Embedding(n_items, K)
        self.W_u      = nn.Linear(K, K, bias=False)
        self.W_v      = nn.Linear(K, K, bias=False)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def aggregate(self, src_emb, src_idx, dst_idx, n_dst, transform):
        """Mean aggregation: average neighbour src embeddings for each dst node."""
        dev = src_emb.weight.device
        K   = src_emb.weight.size(1)
        # FIX 2: create agg/cnt on the same device as the embedding weights
        agg = torch.zeros(n_dst, K, device=dev)
        cnt = torch.zeros(n_dst,    device=dev)
        agg.index_add_(0, dst_idx, src_emb(src_idx))
        # FIX 3: use 1-D dst_idx and clamp AFTER counting
        cnt.index_add_(0, dst_idx, torch.ones(len(dst_idx), device=dev))
        cnt = cnt.clamp(min=1)
        return torch.relu(transform(agg / cnt.unsqueeze(1)))

    def forward(self, u_idx, i_idx, edge_u, edge_i):
        u_agg = self.aggregate(self.item_emb, edge_i, edge_u,
                               self.user_emb.weight.size(0), self.W_u)
        v_agg = self.aggregate(self.user_emb, edge_u, edge_i,
                               self.item_emb.weight.size(0), self.W_v)
        u_h = self.user_emb(u_idx) + u_agg[u_idx]
        v_h = self.item_emb(i_idx) + v_agg[i_idx]
        return torch.sigmoid((u_h * v_h).sum(dim=-1))

    def predict_all(self, edge_u, edge_i):
        """Full (n_users, n_items) prediction matrix for RMSE evaluation."""
        # FIX 4: removed unused all_u / all_i variables
        u_agg = self.aggregate(self.item_emb, edge_i, edge_u,
                               self.user_emb.weight.size(0), self.W_u)
        v_agg = self.aggregate(self.user_emb, edge_u, edge_i,
                               self.item_emb.weight.size(0), self.W_v)
        u_h = self.user_emb.weight + u_agg
        v_h = self.item_emb.weight + v_agg
        return torch.sigmoid(torch.mm(u_h, v_h.t()))


class GNNLoss(nn.Module):
    def __init__(self, lam=0.01):
        super().__init__()
        self.lam = lam

    def forward(self, pred, target, model):
        mse = nn.functional.mse_loss(pred, target)
        reg = self.lam * sum(p.norm() for p in model.parameters())
        return mse + reg


def compute_rmse(matrix, predicted, min_r, max_r):
    mask    = (matrix != -1).float()
    n       = mask.sum()
    if n == 0: return float('nan')
    pred_sc = predicted * (max_r - min_r) + min_r
    true_sc = matrix    * (max_r - min_r) + min_r
    return math.sqrt((((pred_sc - true_sc) ** 2 * mask).sum() / n).item())


print("FedGNN, GNNLoss, compute_rmse defined.")


## 4. Parameter Tuning -- Grid Search

Run **once** after building edge tensors. Writes `lr`, `lam`, `latent_vectors`.


In [ ]:
# ── Grid ─────────────────────────────────────────────────────────────────────
LR_GRID     = [0.001, 0.01,  0.05]
LAM_GRID    = [0.01,  0.1,   0.3 ]
LATENT_GRID = [10,    20,    40  ]
TUNE_EPOCHS = 30
TUNE_PAT    = 5

param_grid = list(itertools.product(LR_GRID, LAM_GRID, LATENT_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>7}  {'lam':>6}  {'K':>4}  {'best val RMSE':>14}")
print("-" * 42)
grid_results = []

for _k, (_lr, _lam, _K) in enumerate(param_grid, 1):
    torch.manual_seed(0)
    _model   = FedGNN(n_users, n_items, _K)
    _loss_fn = GNNLoss(lam=_lam)
    _opt     = torch.optim.Adam(_model.parameters(), lr=_lr)
    _best_val, _wait = float('inf'), 0
    for _ep in range(TUNE_EPOCHS):
        _model.train()
        _opt.zero_grad()
        _pred = _model(train_u, train_i, train_u, train_i)
        _loss = _loss_fn(_pred, train_y, _model)
        _loss.backward()
        _opt.step()
        _model.eval()
        with torch.no_grad():
            _pred_mat = _model.predict_all(train_u, train_i)
            _v = compute_rmse(val_matrix, _pred_mat, min_rating, max_rating)
        if _v < _best_val: _best_val, _wait = _v, 0
        else:
            _wait += 1
            if _wait >= TUNE_PAT: break
    grid_results.append((_best_val, _lr, _lam, _K))
    _marker = ' <--' if _best_val == min(r[0] for r in grid_results) else ''
    print(f"{_k:>3}  {_lr:>7.4f}  {_lam:>6.3f}  {_K:>4d}  {_best_val:>14.4f}{_marker}")

best_val_t, best_lr, best_lam, best_K = min(grid_results, key=lambda x: x[0])
print(f"\nBest val RMSE  : {best_val_t:.4f}")
print(f"  lr             = {best_lr}")
print(f"  lam            = {best_lam}")
print(f"  latent_vectors = {best_K}")
lr             = best_lr
lam            = best_lam
latent_vectors = best_K
print("\nHyperparameters updated. Run the training cell.")


## 5. Training


In [ ]:
try: latent_vectors
except NameError: latent_vectors = 20
try: lr
except NameError: lr = 0.01
try: lam
except NameError: lam = 0.01
num_epoch = 100
patience  = 10

torch.manual_seed(0)
model    = FedGNN(n_users, n_items, latent_vectors)
loss_fn  = GNNLoss(lam=lam)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

best_val_rmse, patience_count, best_state = float('inf'), 0, None

for epoch in range(num_epoch):
    model.train()
    optimizer.zero_grad()
    pred = model(train_u, train_i, train_u, train_i)
    loss = loss_fn(pred, train_y, model)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        pred_mat   = model.predict_all(train_u, train_i)
        train_rmse = compute_rmse(train_matrix, pred_mat, min_rating, max_rating)
        val_rmse   = compute_rmse(val_matrix,   pred_mat, min_rating, max_rating)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {loss.item():.4f} "
              f"| train RMSE: {train_rmse:.4f} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch}.")
            break

model.load_state_dict(best_state)
print(f"\nBest val RMSE: {best_val_rmse:.4f}")


## 6. Test Evaluation


In [ ]:
model.eval()
with torch.no_grad():
    pred_test = model.predict_all(train_u, train_i)
    test_rmse = compute_rmse(test_matrix, pred_test, min_rating, max_rating)
    mask     = (test_matrix != -1).float()
    n_obs    = mask.sum()
    pred_sc  = pred_test   * (max_rating - min_rating) + min_rating
    true_sc  = test_matrix * (max_rating - min_rating) + min_rating
    test_mae = (torch.abs(pred_sc - true_sc) * mask).sum() / n_obs
print(f"Test RMSE : {test_rmse:.4f}")
print(f"Test MAE  : {test_mae.item():.4f}")
